# Faunari · Notebook 01 — Data Ingestion Pipeline

Fetches the Phase-1 snake data from **all finalized sources** into `../data/raw/`, writing
a per-source **provenance manifest** (source, id, species, license, url) to `../data/manifests/`.

> **Governing rule:** track license provenance per source/image now; filter before any public release.

**Sources:** Kaggle India set (Track-1 prototype) · GBIF · iNaturalist · GitHub Indian-Snakes · SnakeCLEF (opt-in).

**Prerequisites**
- `pip install requests` (everything uses `requests` — no extra Kaggle package needed).
- Kaggle API token at `~/.kaggle/kaggle.json` (kaggle.com/settings → Create New API Token) for the Kaggle source.
- `git` on PATH for the GitHub source.

In [1]:
# Standard imports + tunable config kept at the top so the whole pipeline is configurable in one place.
from __future__ import annotations

import csv
import json
import os
import subprocess
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Iterable

import requests

# --- Paths (resolve repo root whether run from Trials/ or the project root) ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Trials" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
MANIFEST_DIR = DATA_DIR / "manifests"

# --- Tunables (small caps keep prototyping fast; raise for the full corpus) ---
REQUEST_TIMEOUT = 30          # seconds per HTTP call
INAT_MAX_RECORDS = 2000       # cap citizen-science pulls during prototyping
GBIF_MAX_RECORDS = 1500
PER_PAGE = 200                # API page size (max for both APIs)
USER_AGENT = "Faunari-data-ingestion/0.1 (research prototype)"

In [2]:
def ensure_dirs(*dirs: Path) -> None:
    """Create directories (and parents) if missing — idempotent so re-runs are safe."""
    for d in dirs:
        d.mkdir(parents=True, exist_ok=True)


ensure_dirs(RAW_DIR, MANIFEST_DIR)
print(f"Data root: {DATA_DIR}")

Data root: e:\Coding_Notes\Faunari\data


## Shared helpers — manifest + streamed download

The manifest is the backbone of our "decide licensing per-source later" policy: every image
we keep is logged with its license and origin so we can filter before release.

In [3]:
MANIFEST_FIELDS = ["source", "record_id", "scientific_name", "license",
                   "image_url", "local_path", "captured_at"]


def write_manifest(source: str, rows: list[dict[str, Any]]) -> Path:
    """Persist per-image provenance (license/source/species) — enables pre-release filtering."""
    path = MANIFEST_DIR / f"{source}_manifest.csv"
    with path.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=MANIFEST_FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    return path


def download_file(url: str, dest: Path, *, chunk: int = 1 << 16) -> Path:
    """Stream a URL to disk so memory stays flat regardless of file size."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": USER_AGENT}
    with requests.get(url, stream=True, timeout=REQUEST_TIMEOUT, headers=headers) as r:
        r.raise_for_status()
        with dest.open("wb") as fh:
            for part in r.iter_content(chunk_size=chunk):
                fh.write(part)
    return dest

## Source 1 — Kaggle "Snake Dataset – India" (Track-1 prototype)

Small, India-specific binary set. Used to stand up the pipeline + baseline + Streamlit UX fast.
Downloaded straight via `requests` using HTTP basic auth from `kaggle.json` — no Kaggle package required.

In [4]:
def _kaggle_credentials() -> tuple[str, str]:
    """Find Kaggle creds from env vars or kaggle.json in common spots; clear error if absent."""
    # 1) Env vars win if set (KAGGLE_USERNAME / KAGGLE_KEY).
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"]
    # 2) Otherwise look for kaggle.json in the usual locations.
    cfg = os.environ.get("KAGGLE_CONFIG_DIR")
    candidates = [
        Path(cfg) / "kaggle.json" if cfg else None,
        Path.home() / ".kaggle" / "kaggle.json",
        PROJECT_ROOT / "kaggle.json",
        Path.home() / "Downloads" / "kaggle.json",
    ]
    for path in filter(None, candidates):
        if path.is_file():
            creds = json.loads(path.read_text(encoding="utf-8"))
            return creds["username"], creds["key"]
    searched = "\n  ".join(str(p) for p in filter(None, candidates))
    raise FileNotFoundError(
        "kaggle.json not found. Create a token at kaggle.com/settings -> API -> "
        "'Create New API Token', then save it to one of:\n  " + searched
    )


def fetch_kaggle_india(dest: Path = RAW_DIR / "kaggle_india") -> Path:
    """Pull the Track-1 baseline set via the Kaggle REST API with requests (no kaggle package)."""
    dest.mkdir(parents=True, exist_ok=True)
    user, key = _kaggle_credentials()
    url = "https://www.kaggle.com/api/v1/datasets/download/adityasharma01/snake-dataset-india"
    archive = dest / "snake-dataset-india.zip"
    with requests.get(url, auth=(user, key), stream=True, timeout=REQUEST_TIMEOUT) as r:
        r.raise_for_status()  # 401/403 here means the token is missing or invalid
        with archive.open("wb") as fh:
            for part in r.iter_content(chunk_size=1 << 16):
                fh.write(part)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(dest)
    archive.unlink()  # drop the zip once extracted to keep the data dir clean
    return dest

## Source 2 — GBIF (India snake occurrences with media)

We resolve the GBIF backbone taxon key at runtime instead of hardcoding a possibly-stale id,
then page occurrences that carry still-image media — logging the license on every record.

In [5]:
GBIF_API = "https://api.gbif.org/v1"


def _gbif_taxon_key(name: str = "Serpentes") -> int:
    """Resolve the GBIF backbone key at runtime so we never depend on a hardcoded id."""
    r = requests.get(f"{GBIF_API}/species/match", params={"name": name}, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    key = r.json().get("usageKey")
    if key is None:
        raise RuntimeError(f"Could not resolve GBIF taxon key for {name!r}")
    return key


def fetch_gbif(country: str = "IN", max_records: int = GBIF_MAX_RECORDS,
               dest: Path = RAW_DIR / "gbif") -> Path:
    """Download India snake occurrence images (one representative per occurrence) + license."""
    taxon_key = _gbif_taxon_key("Serpentes")
    rows: list[dict[str, Any]] = []
    offset = 0
    while offset < max_records:
        r = requests.get(
            f"{GBIF_API}/occurrence/search",
            params={"taxonKey": taxon_key, "country": country, "mediaType": "StillImage",
                    "limit": PER_PAGE, "offset": offset},
            timeout=REQUEST_TIMEOUT,
        )
        r.raise_for_status()
        payload = r.json()
        results = payload.get("results", [])
        if not results:
            break
        for rec in results:
            for media in (rec.get("media") or []):
                img_url = media.get("identifier")
                if not img_url:
                    continue
                rid = str(rec.get("key"))
                ext = Path(img_url.split("?")[0]).suffix or ".jpg"
                local = dest / f"{rid}{ext}"
                try:
                    download_file(img_url, local)
                except Exception as exc:  # noqa: BLE001 - skip unreachable media, keep going
                    print(f"  skip {img_url}: {exc}")
                    continue
                rows.append({"source": "gbif", "record_id": rid,
                             "scientific_name": rec.get("scientificName"),
                             "license": rec.get("license") or media.get("license"),
                             "image_url": img_url, "local_path": str(local),
                             "captured_at": rec.get("eventDate")})
                break  # one image per occurrence keeps the pull lean and diverse
        offset += PER_PAGE
        if payload.get("endOfRecords"):
            break
        time.sleep(0.2)  # be polite to the API
    write_manifest("gbif", rows)
    print(f"GBIF: saved {len(rows)} images -> {dest}")
    return dest

## Source 3 — iNaturalist (research-grade India snake photos)

**Fixed:** filter by the Serpentes **taxon_id** (resolved at runtime), not `taxon_name`. The
name-match query was contaminated with non-snakes (serpent *eagles*, darters, snakehead *fish*).
Marine sea-snakes are excluded (out of scope, BRD §5.2). Each photo's `license_code` is captured.

In [6]:
INAT_API = "https://api.inaturalist.org/v1"

# Marine sea-snakes are out of scope (BRD section 5.2) — coarse genus-prefix exclusion.
INAT_EXCLUDE_PREFIXES = ("Hydrophis", "Enhydrina", "Pelamis", "Microcephalophis", "Laticauda")


def _inat_place_id(name: str = "India") -> int:
    """Resolve the iNaturalist place id at runtime to avoid stale hardcoded ids."""
    r = requests.get(f"{INAT_API}/places/autocomplete", params={"q": name},
                     timeout=REQUEST_TIMEOUT, headers={"User-Agent": USER_AGENT})
    r.raise_for_status()
    results = r.json().get("results", [])
    if not results:
        raise RuntimeError(f"No iNaturalist place found for {name!r}")
    return results[0]["id"]


def _inat_taxon_id(name: str = "Serpentes", rank: str = "suborder") -> int:
    """Resolve the Serpentes taxon id so we filter by TAXONOMY, not a name string-match.

    The old `taxon_name=Serpentes` matched names (serpent eagles etc.); taxon_id returns only
    true snakes (the suborder and all its descendants).
    """
    r = requests.get(f"{INAT_API}/taxa", params={"q": name, "rank": rank},
                     timeout=REQUEST_TIMEOUT, headers={"User-Agent": USER_AGENT})
    r.raise_for_status()
    for res in r.json().get("results", []):
        if res.get("name") == name and res.get("rank") == rank:
            return res["id"]
    raise RuntimeError(f"Could not resolve iNaturalist taxon id for {name!r} ({rank})")


def _is_marine(scientific_name: str | None) -> bool:
    """Skip out-of-scope marine sea-snakes by genus prefix (coarse but effective)."""
    return bool(scientific_name) and scientific_name.startswith(INAT_EXCLUDE_PREFIXES)


def fetch_inaturalist(max_records: int = INAT_MAX_RECORDS,
                      dest: Path = RAW_DIR / "inaturalist") -> Path:
    """Download research-grade India *snake* photos (filtered by Serpentes taxon_id) + license."""
    place_id = _inat_place_id("India")
    taxon_id = _inat_taxon_id("Serpentes")
    rows: list[dict[str, Any]] = []
    page, fetched = 1, 0
    while fetched < max_records:
        r = requests.get(
            f"{INAT_API}/observations",
            params={"taxon_id": taxon_id, "place_id": place_id,
                    "quality_grade": "research", "photos": "true",
                    "per_page": PER_PAGE, "page": page},
            timeout=REQUEST_TIMEOUT, headers={"User-Agent": USER_AGENT},
        )
        r.raise_for_status()
        results = r.json().get("results", [])
        if not results:
            break
        for obs in results:
            taxon = obs.get("taxon") or {}
            if _is_marine(taxon.get("name")):
                continue  # out of scope (marine)
            photos = obs.get("photos") or []
            if not photos:
                continue
            img_url = (photos[0].get("url") or "").replace("square", "medium")  # bigger usable size
            if not img_url:
                continue
            rid = str(obs.get("id"))
            local = dest / f"{rid}.jpg"
            try:
                download_file(img_url, local)
            except Exception as exc:  # noqa: BLE001
                print(f"  skip {img_url}: {exc}")
                continue
            rows.append({"source": "inaturalist", "record_id": rid,
                         "scientific_name": taxon.get("name"),
                         "license": photos[0].get("license_code"),
                         "image_url": img_url, "local_path": str(local),
                         "captured_at": obs.get("observed_on")})
            fetched += 1
            if fetched >= max_records:
                break
        page += 1
        time.sleep(0.5)  # respect iNat rate limits
    write_manifest("inaturalist", rows)
    print(f"iNaturalist: saved {len(rows)} snake images -> {dest}")
    return dest

## Source 4 — GitHub: arjun921/Indian-Snakes-Dataset (supplement)

Shallow clone keeps it fast and small.

In [7]:
def fetch_github_indian_snakes(dest: Path = RAW_DIR / "github_indian_snakes") -> Path:
    """Shallow-clone the supplementary Indian-subcontinent set (fast, no history)."""
    if dest.exists():
        print(f"GitHub set already present at {dest}")
        return dest
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/arjun921/Indian-Snakes-Dataset.git", str(dest)],
        check=True,
    )
    return dest

## Source 5 — SnakeCLEF 2024 (opt-in, large)

The big, species-rich set with WHO medically-important mapping. The download is large and the
URL changes, so pass the **current** archive link (grab it from
[imageclef.org/node/319](https://www.imageclef.org/node/319) — start with the **240px** training
archive, ~4.5 GB). Disabled by default in the run cell below.

In [8]:
def fetch_snakeclef(archive_url: str, dest: Path = RAW_DIR / "snakeclef") -> Path:
    """Download + extract a SnakeCLEF archive by URL (use the 240px set for tractable prototyping)."""
    dest.mkdir(parents=True, exist_ok=True)
    archive = dest / Path(archive_url.split("?")[0]).name
    download_file(archive_url, archive)
    if archive.suffix == ".zip":
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(dest)
    elif archive.name.endswith((".tar.gz", ".tgz")):
        import tarfile
        with tarfile.open(archive) as tf:
            tf.extractall(dest)
    return dest

## Run the pipeline

Each source runs independently — one failure (e.g. missing Kaggle token) is reported but never
aborts the others. Toggle `enabled` per source as needed.

In [9]:
@dataclass
class SourceSpec:
    """A named, toggleable ingestion step — decouples *what* runs from *how* it runs (SRP)."""
    name: str
    fn: Callable[[], Path]
    enabled: bool = True


def run_ingestion(sources: Iterable[SourceSpec]) -> dict[str, str]:
    """Run each enabled source in isolation so a single failure never aborts the batch."""
    summary: dict[str, str] = {}
    for spec in sources:
        if not spec.enabled:
            summary[spec.name] = "skipped"
            continue
        try:
            summary[spec.name] = f"ok -> {spec.fn()}"
        except Exception as exc:  # noqa: BLE001 - report and continue with the next source
            summary[spec.name] = f"FAILED: {exc}"
    return summary

In [10]:
# Re-pull iNaturalist with the FIXED taxon_id query (the old name-match pull was contaminated
# with birds/fish). Completed sources are commented out; GBIF is dropped for Phase 1.
SOURCES = [
    # SourceSpec("kaggle_india", fetch_kaggle_india, enabled=True),                  # done
    SourceSpec("inaturalist", fetch_inaturalist, enabled=True),                       # RE-PULL (fixed)
    # SourceSpec("github_indian_snakes", fetch_github_indian_snakes, enabled=True),   # done (cloned)
    # GBIF dropped for Phase 1 - flaky media, 1 usable image (see docs/PROJECT_PLAN.md).
    # SnakeCLEF: paste the current 240px URL from imageclef.org/node/319, then enable.
    # SourceSpec("snakeclef", lambda: fetch_snakeclef("<paste-url>"), enabled=False),
]

results = run_ingestion(SOURCES)
print("\n=== Ingestion summary ===")
for name, status in results.items():
    print(f"{name:24s} {status}")


=== Ingestion summary ===
kaggle_india             ok -> e:\Coding_Notes\Faunari\data\raw\kaggle_india
